## Introduction to Pydantic AI

## Load Environment Variables

In [1]:
from dotenv import load_dotenv

load_dotenv("Docker.env")

True

## Define LLM Provider

In [2]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.azure import AzureProvider
from pydantic_ai.settings import ModelSettings
import os

provider = AzureProvider(
    azure_endpoint=os.getenv("LLM__AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("LLM__AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("LLM__AZURE_OPENAI_API_VERSION"),
)

model = OpenAIChatModel(os.getenv("LLM__MODEL", ""), provider=provider)
model_settings = ModelSettings(
    temperature=float(os.getenv("LLM__TEMPERATURE", "0.1")),
    max_tokens=int(os.getenv("LLM__MAX_TOKENS", "1000")),
    top_p=float(os.getenv("LLM__TOP_P", "0.95")),
)

## Simple LLM communication with LLM

note:
- The pydantic-ai docs explicitly recommend using `instructions` over `system_prompt` for most cases. The difference is that instructions are always re-applied fresh per run, while `system_prompt` gets carried across message history. Since we're doing single-turn calls, `instructions` is the right choice.
- In pydantic-ai, `Agent` is their primary interface for all LLM interactions — even simple single-turn calls. So Agent is the correct and simplest way to get output from pydantic-ai, regardless of whether the use case is agentic or not.

In [3]:
from pydantic_ai import Agent

instruction_prompt = (
    "You are a helpful assistant. You always answer with a single concise sentence."
)

agent = Agent(
    model=model,
    instructions=instruction_prompt,
    retries=5,
)

user_prompt = "what is Python?"
response = await agent.run(user_prompt)
response.output

'Python is a popular high-level programming language known for its readability, versatility, and wide use in web development, data science, automation, and more.'

## Message History

In [4]:
from pydantic_ai import Agent

instruction_prompt = (
    "You are a helpful assistant. You always answer with a single concise sentence."
)

agent = Agent(
    model=model,
    instructions=instruction_prompt,
    retries=5,
)

In [5]:
user_prompt = "What is Python?"
response = await agent.run(user_prompt)
response.output

'Python is a high-level, versatile programming language known for its readability and wide use in web development, data science, automation, and more.'

In [6]:
user_prompt = "And when was it released?"
response = await agent.run(user_prompt)
# response = await agent.run(user_prompt, message_history=response.all_messages())
response.output

'I’m not sure what “it” refers to—please tell me the product, movie, song, or app you mean.'

In [7]:
# response.all_messages()
# response.new_messages()

## Structured Output

In [8]:
from pydantic import BaseModel


class Person(BaseModel):
    name: str
    age: int
    job: str

In [9]:
from pydantic_ai.run import AgentRunResult
from pydantic_ai import Agent

instruction_prompt = (
    "You are a helpful assistant. You always answer with a structured output of person."
)


agent = Agent(
    model=model,
    instructions=instruction_prompt,
    retries=5,
    output_type=Person,
)

user_prompt = "Jhon is a 32 years old Police Officer"
response = await agent.run(user_prompt)
response.output

Person(name='Jhon', age=32, job='Police Officer')

In [10]:
type(response.output)

__main__.Person

## Agentic AI with Pydantic AI

### The need

In [11]:
from pydantic_ai import Agent

instruction_prompt = "You are a helpful assistant."

agent = Agent(
    model=model,
    instructions=instruction_prompt,
    retries=5,
)

user_prompt = "what is my favorite color?"
response = await agent.run(user_prompt)
response.output

'I don’t know yet — you haven’t told me your favorite color. If you want, tell me and I’ll remember it for this conversation.'

## Agentic AI Tools - style 1

In [12]:
def get_favorite_color(name: str) -> str:
    """Get the favorite color of a person"""
    if name == "John":
        return "Blue"
    elif name == "Jane":
        return "Red"
    else:
        return "Unknown"

In [13]:
from pydantic_ai import Agent

system_prompt = "You are a helpful assistant."

agent = Agent(
    model=model,
    instructions=system_prompt,
    retries=5,
    tools=[get_favorite_color],
)

user_prompt = "Hi I am Jane, what is my favorite color?"
response = await agent.run(user_prompt)
response.output

'Hi Jane — your favorite color is **Red**.'

## Agentic AI Tools - style 2

In [14]:
from pydantic_ai import Agent

system_prompt = "You are a helpful assistant."

agent = Agent(
    model=model,
    instructions=system_prompt,
    retries=5,
)

In [15]:
@agent.tool_plain
def get_favorite_color(name: str) -> str:
    """Get the favorite color of a person"""
    if name == "John":
        return "Blue"
    elif name == "Jane":
        return "Red"
    else:
        return "Unknown"

In [16]:
user_prompt = "Hi I am John, what is my favorite color?"
response = await agent.run(user_prompt)
response.output

'Hi John — your favorite color is **blue**.'

## Dependency Injection and Context

### Case 1: Adding context for the tool

In [17]:
from pydantic_ai import Agent

system_prompt = "You are a roulette assistant. You can process number guesses from users and return if the guess is correct or not."

agent = Agent(
    model=model,
    instructions=system_prompt,
    retries=5,
    deps_type=int,
    output_type=bool,
)

In [18]:
from pydantic_ai import RunContext


@agent.tool
async def guess_roulette_number(ctx: RunContext, guessed_number: int) -> str:
    """Guess a roulette number for the user"""
    return "won" if guessed_number == ctx.deps else "lost"

In [19]:
correct_number = 42
user_prompt = "I am guessing the number is eighteen"
response = await agent.run(user_prompt, deps=correct_number)
response.output

False

In [20]:
correct_number = 42
user_prompt = "I am guessing the number is  fourty two"
response = await agent.run(user_prompt, deps=correct_number)
response.output

True

### Case 2: Adding context to the system prompt

In [21]:
import sqlite3
from dataclasses import dataclass


@dataclass
class DBDependency:
    user_name: str
    db_connection: sqlite3.Connection


@dataclass
class User:
    id: int
    name: str
    email: str

In [22]:
from pydantic_ai import Agent

system_prompt = "You are a helpful assistant. Always start every message by calling the user by name."

agent = Agent(
    model=model,
    instructions=system_prompt,
    retries=5,
    deps_type=DBDependency,
)

In [23]:
from pydantic_ai import RunContext


@agent.system_prompt
def add_user_name(ctx: RunContext[DBDependency]) -> str:
    return f"The user name is {ctx.deps.user_name}."


@agent.tool
def list_all_employees(ctx: RunContext[DBDependency]) -> list[tuple]:
    """List all the employees from the database."""
    conn = ctx.deps.db_connection
    cursor = conn.cursor()
    cursor.execute("SELECT name, age, profession FROM person;")
    people = cursor.fetchall()
    result = []
    for row in people:
        result.append((row[0], row[1], row[2]))
    return result

In [24]:
db_conn = sqlite3.connect("database.db", check_same_thread=False)
user_name = "boedy"
deps = DBDependency(user_name, db_conn)
response = await agent.run("What employees are in the DB?", deps=deps)
print(response.output)

boedy, the employees in the database are:

- Mike — 20 — Plumber
- Lisa — 30 — Accountant
- John — 40 — Programmer


## Toolsets

In [25]:
from datetime import datetime
from pydantic_ai.toolsets import FunctionToolset


def get_current_date_tool() -> str:
    today = datetime.today()
    return today.strftime("%Y-%m-%d")


def get_current_weekday_tool() -> str:
    today = datetime.today()
    return today.strftime("%A")


def get_current_time_tool() -> str:
    now = datetime.now()
    return now.strftime("%H:%M:%S")


date_time_toolset = FunctionToolset(
    tools=[get_current_date_tool, get_current_weekday_tool, get_current_time_tool],
)


In [ ]:
from pydantic_ai import Agent

system_prompt = "You are a helpful assistant."

agent = Agent(
    model=model, instructions=system_prompt, retries=5, toolsets=[date_time_toolset]
)

user_prompt = "What is the date and time?"
response = await agent.run(user_prompt)
# response = await agent.run(user_prompt, toolsets=[date_time_toolset])

response.output

'The current date is **2026-04-15** and the current time is **19:34:14**.'